## Borrado Librerias

In [12]:
#RAW DATA
import os

# Carpeta que vaciar
folder_path = r"C:\Users\carlo\Documents\TFM\01 Datasets\00 Raw Data"
#folder_path = r"C:\Users\carlo\Documents\TFM\01 Datasets\01 Grouped Data"
#folder_path = r"C:\Users\carlo\Documents\TFM\01 Datasets\02 Full Dataset"

for filename in os.listdir(folder_path):
    file_path = os.path.join(folder_path, filename)
    if os.path.isfile(file_path):
        os.remove(file_path)
        print(f"🗑️ Borrado: {file_path}")

## Importaciones

In [5]:
import os
import re
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm
import pandas as pd

In [16]:
pip install pandas pyarrow

Note: you may need to restart the kernel to use updated packages.


## Descarga de datos:
Link: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

In [ ]:
# URL de la página
url = "https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page"

#Carpeta Destino
output_dir = r"C:\Users\carlo\Documents\TFM\01 Datasets\00 Raw Data"
os.makedirs(output_dir, exist_ok=True)

# Rango de años
start_year = 2021
end_year = 2025

# Obtener HTML de la página
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

# Buscar todos los enlaces a archivos .parquet
links = soup.find_all("a", href=True)
parquet_links = [
    a["href"] for a in links
    if ".parquet" in a["href"].lower()
]

# Filtrar solo los años deseados
pattern = re.compile(r"(\d{4})-(\d{2})\.parquet", re.IGNORECASE)
filtered_links = []

for link in parquet_links:
    match = pattern.search(link)
    if match:
        year = int(match.group(1))
        if start_year <= year <= end_year:
            # Asegurar URL completa
            if link.startswith("/"):
                full_url = "https://www.nyc.gov" + link
            else:
                full_url = link
            filtered_links.append(full_url)

# Descargar archivos
print(f"Se encontraron {len(filtered_links)} archivos. Iniciando descarga...\n")

for file_url in tqdm(filtered_links):
    # Strip any trailing whitespace from the URL
    clean_file_url = file_url.strip()
    filename = os.path.basename(clean_file_url)
    dest_path = os.path.join(output_dir, filename)

    if not os.path.exists(dest_path):
        # Use the cleaned URL for the request
        with requests.get(clean_file_url, stream=True) as r:
            r.raise_for_status()
            with open(dest_path, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
    else:
        print(f"{filename} ya existe, se omite.")

print("\n✅ Descarga completada.")

Se encontraron 204 archivos. Iniciando descarga...



 49%|███████████████████████████████████████▎                                         | 99/204 [07:35<04:25,  2.53s/it]

## Combinar datos por categoria y Año

## Combinar datos por categoria

In [7]:
#Si lo quiero desde drive
data_dir = r"C:\Users\carlo\Documents\TFM\01 Datasets\00 Raw Data"

#Si lo si quisiera acumular en drive:
output_dir = r"C:\Users\carlo\Documents\TFM\01 Datasets\01 Grouped Data"

os.makedirs(output_dir, exist_ok=True)

# 🗓️ Rango de años que quieres procesar
year_min = 2021
year_max = 2025

# 🔧 Formato de salida
output_format = "parquet"

# Categorías y sus archivos
categories = {
    "yellow": [],
    "green": [],
    "fhv": [],
    "hvfhv": []
}

# Clasificar los archivos por tipo
for file in os.listdir(data_dir):
    lower = file.lower()
    path = os.path.join(data_dir, file)

    if "yellow" in lower:
        categories["yellow"].append(path)
    elif "green" in lower:
        categories["green"].append(path)
    elif "fhv" in lower and "hvfhv" not in lower:
        categories["fhv"].append(path)
    elif "hvfhv" in lower:
        categories["hvfhv"].append(path)

# Regex para extraer año y mes
pattern = re.compile(r"(\d{4})-(\d{2})")

# Procesar cada categoría
for cat, files in categories.items():
    print(f"\n📦 Procesando categoría: {cat}")
    dfs = []

    for file in tqdm(sorted(files)):
        match = pattern.search(file)
        if not match:
            print(f"⚠️ Año y mes no encontrados en {file}")
            continue

        year, month = map(int, match.groups())

        # 👉 FILTRO POR RANGO DE AÑOS
        if not (year_min <= year <= year_max):
            continue

        try:
            df = pd.read_parquet(file)
            df["year"] = year
            df["month"] = month
            dfs.append(df)

            # Eliminar el archivo original
            ##os.remove(file)

        except Exception as e:
            print(f"❌ Error procesando {file}: {e}")

    if dfs:
        full_df = pd.concat(dfs, ignore_index=True)
        output_path = os.path.join(output_dir, f"{cat}_tripdata.{output_format}")

        if output_format == "csv":
            full_df.to_csv(output_path, index=False)
        else:
            full_df.to_parquet(output_path, index=False)

        print(f"✅ Guardado: {output_path}")
    else:
        print(f"⚠️ No hay datos dentro del rango {year_min}–{year_max} para {cat}")


📦 Procesando categoría: yellow


100%|██████████████████████████████████████████████████████████████████████████████████| 51/51 [00:16<00:00,  3.00it/s]


✅ Guardado: C:\Users\carlo\Documents\TFM\01 Datasets\01 Grouped Data\yellow_tripdata.parquet

📦 Procesando categoría: green


100%|██████████████████████████████████████████████████████████████████████████████████| 51/51 [00:01<00:00, 45.04it/s]


✅ Guardado: C:\Users\carlo\Documents\TFM\01 Datasets\01 Grouped Data\green_tripdata.parquet

📦 Procesando categoría: fhv


...

 99%|███████████████████████████████████████████████████████████████████████████████▏| 101/102 [07:26<00:20, 20.51s/it]

❌ Error procesando C:\Users\carlo\Documents\TFM\01 Datasets\00 Raw Data\fhvhv_tripdata_2025-02.parquet: Unable to allocate 1.15 GiB for an array with shape (8, 19339461) and data type object


100%|████████████████████████████████████████████████████████████████████████████████| 102/102 [07:45<00:00,  4.56s/it]

❌ Error procesando C:\Users\carlo\Documents\TFM\01 Datasets\00 Raw Data\fhvhv_tripdata_2025-03.parquet: malloc of size 8388608 failed


MemoryError: Unable to allocate 4.39 GiB for an array with shape (1, 589823321) and data type object

## Combinamos las 4 categorias en 1

In [ ]:
# 📂 Carpeta de entrada (donde están los archivos individuales)
input_dir = r"C:\Users\carlo\Documents\TFM\01 Datasets\01 Grouped Data"

# 📂 Carpeta de salida (donde irá el archivo combinado final)
final_dir = r"C:\Users\carlo\Documents\TFM\01 Datasets\02 Full Dataset"
os.makedirs(final_dir, exist_ok=True)

# 📄 Nombre del archivo final
output_file = os.path.join(final_dir, "all_dataset.parquet")

# Categorías y nombres de archivos
categories = {
    "yellow": "yellow_tripdata.parquet",
    "green": "green_tripdata.parquet",
    "fhv": "fhv_tripdata.parquet",
    "hvfhv": "hvfhv_tripdata.parquet"
}

dfs = []

for cat, filename in categories.items():
    path = os.path.join(input_dir, filename)

    if not os.path.exists(path):
        print(f"⚠️ Archivo no encontrado: {path}")
        continue

    try:
        df = pd.read_parquet(path)
        df["categoria"] = cat
        dfs.append(df)
        print(f"✅ Cargado: {filename} ({len(df)} filas)")
    except Exception as e:
        print(f"❌ Error leyendo {filename}: {e}")

# Concatenar y guardar
if dfs:
    all_data = pd.concat(dfs, ignore_index=True)
    all_data.to_parquet(output_file, index=False)
    print(f"\n✅ Archivo combinado guardado en: {output_file}")

"""
    # Borrar los archivos individuales
    for filename in categories.values():
        path = os.path.join(input_dir, filename)
        try:
            os.remove(path)
            print(f"🗑️ Borrado: {filename}")
        except Exception as e:
            print(f"⚠️ No se pudo borrar {filename}: {e}")
"""
else:
    print("⚠️ No se cargaron datos. Verifica los archivos.")


In [ ]:
\*